# Task 1 – Focal Terms (20260323)

For each patent in `SampleGloria_Pat_GlinerLabels_20260323.parquet`:
1. Compile the set of terms appearing in the patent.
2. Compile the set of terms appearing in its cited papers (via `SampleGloria_Link_PmidOa_20260323.parquet` → `SampleGloria_Pmed_GlinerLabels_20260323.parquet`).
3. Identify the intersection → **focal terms**.
4. Record frequency in patent and frequency in cited papers.

**Output:** `focal_terms_20260323.parquet`

In [13]:
import pandas as pd

# Load data
pat  = pd.read_parquet('../data/raw/SampleGloria_Pat_GlinerLabels_20260323.parquet')
link = pd.read_parquet('../data/raw/SampleGloria_Link_PmidOa_20260323.parquet')
pmed = pd.read_parquet('../data/raw/SampleGloria_Pmed_GlinerLabels_20260323.parquet')

print('PAT  shape:', pat.shape,  '| unique patents:', pat['patent_id'].nunique())
print('LINK shape:', link.shape, '| unique patents:', link['patent_id'].nunique(), '| unique pmids:', link['pmid'].nunique())
print('PMED shape:', pmed.shape, '| unique pmids:  ', pmed['pmid'].nunique())

PAT  shape: (16894907, 8) | unique patents: 180000
LINK shape: (2841978, 7) | unique patents: 129467 | unique pmids: 833611
PMED shape: (510752, 4) | unique pmids:   177916


In [14]:
# Step 1: frequency of each term per patent
pat_freq = (
    pat.groupby(['patent_id', 'term'], sort=False)
       .size()
       .reset_index(name='freq_in_patent')
)
print(f'Patent term rows: {len(pat_freq)}')

Patent term rows: 13861510


In [15]:
# Step 2: join link -> pmed to get (patent_id, term) for all cited papers
# drop_duplicates() removes duplicate (patent_id, pmid) rows that arise
# when the same paper was matched via multiple sources in the link file
cited = (
    link[['patent_id', 'pmid']]
    .drop_duplicates()
    .merge(pmed[['pmid', 'term']], on='pmid', how='inner')
)
print(f'Cited (patent_id, pmid, term) rows: {len(cited)}')

Cited (patent_id, pmid, term) rows: 1152893


In [16]:
# Step 3: frequency of each term per patent across all its cited papers
cited_freq = (
    cited.groupby(['patent_id', 'term'], sort=False)
         .size()
         .reset_index(name='freq_in_cited_papers')
)
print(f'Cited term rows: {len(cited_freq)}')

Cited term rows: 932374


In [17]:
# Step 4: intersection = inner join on (patent_id, term)
focal = pat_freq.merge(cited_freq, on=['patent_id', 'term'], how='inner')
focal = focal.rename(columns={'term': 'focal_term'})

print(f'Focal term rows:  {len(focal)}')
print(f'Unique patents:   {focal["patent_id"].nunique()}')
print(f'Unique terms:     {focal["focal_term"].nunique()}')
focal.head(10)

Focal term rows:  19145
Unique patents:   14237
Unique terms:     4611


,patent_id,focal_term,freq_in_patent,freq_in_cited_papers
0,9949959,pirfenidone,1,1
1,10145926,imaging,3,12
2,9869450,50,1,1
3,10076512,lonafarnib,2,3
4,10076512,ritonavir,2,2
5,10076512,pegylated,1,2
6,10005785,metabotropic,1,2
7,10117884,probiotic,2,2
8,10155984,sequencing,2,23
9,10155984,genotyping,1,9


In [18]:
# Save output
focal.to_parquet('../output/focal_terms_20260323.parquet', index=False)
print('Saved: focal_terms_20260323.parquet')

Saved: focal_terms_20260323.parquet
